In [1]:
import yfinance as yf
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [2]:
tickers = ["CMG", "BB", "M", "QQQ"]
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    prices[ticker_symbol] = ticker.history(start="2009-02-14", end="2020-06-11")
    print(f"\n{ticker_symbol}:")
    print(prices[ticker_symbol])


CMG:
                                Open       High        Low      Close  \
Date                                                                    
2009-02-17 00:00:00-05:00   1.097600   1.098800   1.057000   1.067200   
2009-02-18 00:00:00-05:00   1.069800   1.093000   1.048600   1.089000   
2009-02-19 00:00:00-05:00   1.106200   1.120000   1.068800   1.074800   
2009-02-20 00:00:00-05:00   1.054000   1.118000   1.048200   1.110000   
2009-02-23 00:00:00-05:00   1.121400   1.134200   1.086000   1.093400   
...                              ...        ...        ...        ...   
2020-06-04 00:00:00-04:00  20.886200  21.196199  20.700001  20.831200   
2020-06-05 00:00:00-04:00  21.000000  21.382000  20.901800  21.069201   
2020-06-08 00:00:00-04:00  21.056601  21.299999  20.664400  20.984600   
2020-06-09 00:00:00-04:00  20.738199  21.059799  20.700001  20.789400   
2020-06-10 00:00:00-04:00  20.852400  20.858801  20.469801  20.579201   

                             Volume  Dividen

In [ ]:
#Collect Features, we want them a day behind, we give yesterdays stats to guess today's price, so score follows current day
for ticker_symbol in tickers:
    price = prices[ticker_symbol]
    price = price.reset_index()
    price = price.rename(columns={'Date': 'date'})
    sentiment = pd.read_csv(f"../Sentiment/{ticker_symbol}_Sentiment.csv")
    price['date'] = pd.to_datetime(price['date'], utc=True).dt.tz_localize(None).dt.normalize()
    sentiment['date'] = pd.to_datetime(sentiment['date'])
    price = price.merge(sentiment, on='date', how='left')
    price = price.fillna(0)
    print(price)

    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["Score"]       = (price["Close"] > price["Open"]).astype(int)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)
    #Shift the sentiment data back 1 day so that yesturdays headlines affect todays trades
    price["sentiment"] = price["sentiment"].shift(1)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price
    # print(f"\n{ticker_symbol}:")
    # print(price["5DayPerf"])

           date       Open       High        Low      Close    Volume  \
0    2009-02-17   1.097600   1.098800   1.057000   1.067200  32875000   
1    2009-02-18   1.069800   1.093000   1.048600   1.089000  29960000   
2    2009-02-19   1.106200   1.120000   1.068800   1.074800  22835000   
3    2009-02-20   1.054000   1.118000   1.048200   1.110000  46060000   
4    2009-02-23   1.121400   1.134200   1.086000   1.093400  43895000   
...         ...        ...        ...        ...        ...       ...   
2844 2020-06-04  20.886200  21.196199  20.700001  20.831200  17970000   
2845 2020-06-05  21.000000  21.382000  20.901800  21.069201  18545000   
2846 2020-06-08  21.056601  21.299999  20.664400  20.984600  15805000   
2847 2020-06-09  20.738199  21.059799  20.700001  20.789400  13795000   
2848 2020-06-10  20.852400  20.858801  20.469801  20.579201  16105000   

      Dividends  Stock Splits  sentiment  
0           0.0           0.0        0.0  
1           0.0           0.0        

In [4]:

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf", "sentiment"]
all_results = {}

for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    #5 Year window, so the window starts at 2013-06-11 and gets a day added to it as the window changes
    priceTrain = price[(price['date'] <= pd.to_datetime("2018-06-11")) & (price['date'] >= pd.to_datetime("2013-06-11"))]
    priceTest = price[price['date'] > pd.to_datetime("2018-06-11")]


    results = []

    for i in range(len(priceTest)):
        xTrain = priceTrain[features]
        yTrain = priceTrain["Score"]

        scaler = StandardScaler()
        xTrain = scaler.fit_transform(xTrain)

        model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1000, random_state=42)
        model.fit(xTrain, yTrain)

        # Grab yesterdays information to predict today's, data already transformed so i is yesterday
        yest = priceTest.iloc[[i]]
        yestScal = scaler.transform(yest[features])
        pred = model.predict(yestScal)[0]
        prob = model.predict_proba(yestScal)[0][1]
        print("Analyzing Day: ", priceTest.index[i])

        results.append({"Date": priceTest.index[i],
                        "Score": priceTest["Score"].iloc[i],
                        "Prediction": pred,
                        "Probability": prob,
                        "Open": priceTest["Open"].iloc[i],
                        "Close": priceTest["Close"].iloc[i]})

        #First we add the data from the day we just predicted, then we drop the 5 years + 1 day date to keep the window at 5 years
        priceTrain = pd.concat([priceTrain, yest])
        priceTrain = priceTrain.iloc[1:]

    all_results[ticker_symbol] = pd.DataFrame(results)

Analyzing Day:  2346
Analyzing Day:  2347
Analyzing Day:  2348
Analyzing Day:  2349
Analyzing Day:  2350
Analyzing Day:  2351
Analyzing Day:  2352
Analyzing Day:  2353
Analyzing Day:  2354
Analyzing Day:  2355
Analyzing Day:  2356
Analyzing Day:  2357
Analyzing Day:  2358
Analyzing Day:  2359
Analyzing Day:  2360
Analyzing Day:  2361
Analyzing Day:  2362
Analyzing Day:  2363
Analyzing Day:  2364
Analyzing Day:  2365
Analyzing Day:  2366
Analyzing Day:  2367
Analyzing Day:  2368
Analyzing Day:  2369
Analyzing Day:  2370
Analyzing Day:  2371
Analyzing Day:  2372
Analyzing Day:  2373
Analyzing Day:  2374
Analyzing Day:  2375
Analyzing Day:  2376
Analyzing Day:  2377
Analyzing Day:  2378
Analyzing Day:  2379
Analyzing Day:  2380
Analyzing Day:  2381
Analyzing Day:  2382
Analyzing Day:  2383
Analyzing Day:  2384
Analyzing Day:  2385
Analyzing Day:  2386
Analyzing Day:  2387
Analyzing Day:  2388
Analyzing Day:  2389
Analyzing Day:  2390
Analyzing Day:  2391
Analyzing Day:  2392
Analyzing Day

In [5]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)

    print(f"\n{ticker_symbol}:")
    print(" Mean:",results["Score"].mean(), "Accuracy: ", accuracy, "\n  Precision: ", precision, "\n  Recall: ", recall, "\n  F1: ", f1)



CMG:
 Mean: 0.5009940357852882 Accuracy:  0.49105367793240556 
  Precision:  0.4918032786885246 
  Recall:  0.47619047619047616 
  F1:  0.48387096774193544

BB:
 Mean: 0.46322067594433397 Accuracy:  0.5188866799204771 
  Precision:  0.48 
  Recall:  0.463519313304721 
  F1:  0.47161572052401746

M:
 Mean: 0.46322067594433397 Accuracy:  0.5049701789264414 
  Precision:  0.45604395604395603 
  Recall:  0.3562231759656652 
  F1:  0.39999999999999997

QQQ:
 Mean: 0.5308151093439364 Accuracy:  0.4592445328031809 
  Precision:  0.48868778280542985 
  Recall:  0.4044943820224719 
  F1:  0.4426229508196722


# Comments

QQQ again comes out on top with the best F1 (0.60) though accuracy is still below 50%. BB recall is again very low (0.21), barely predicting any up days. CMG and M are in the middle across the board. MLP doesn't show a clear improvement over log reg on these features, which isn't surprising — the bottleneck is the data not the model. Sentiment features will be the real test.

In [6]:
#Find out how much you would have made/lost with this method
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    totalIn = totalOut = totalInvested = totalStoodOut = alltradein = alltradeout = perftradein = perftradeout = 0

    for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
        if i == 1:
            totalIn += 100
            totalInvested += 1
            temp = ((c - o)/o) * 100
            totalOut += 100 + temp
        else:
            totalStoodOut += 1
        alltradein += 100
        alltradeout += (((c-o)/o) * 100) + 100
        if s == 1:
            perftradein += 100
            perftradeout += (((c-o)/o) * 100) + 100

    print(f"\n{ticker_symbol}:")
    print(f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped)")
    print(f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, Profit ${alltradeout - alltradein:.2f}")
    print(f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, Profit ${perftradeout - perftradein:.2f}")


CMG:
  Model:    Invested $24400, Returned $24431.61, Profit $31.61 (244 trades, 259 skipped)
  Buy&Hold: Invested $50300, Returned $50346.79, Profit $46.79
  Perfect:  Invested $25200, Returned $25556.07, Profit $356.07

BB:
  Model:    Invested $22500, Returned $22457.69, Profit $-42.31 (225 trades, 278 skipped)
  Buy&Hold: Invested $50300, Returned $50223.65, Profit $-76.35
  Perfect:  Invested $23300, Returned $23692.35, Profit $392.35

M:
  Model:    Invested $18200, Returned $18171.46, Profit $-28.54 (182 trades, 321 skipped)
  Buy&Hold: Invested $50300, Returned $50171.24, Profit $-128.76
  Perfect:  Invested $23300, Returned $23774.62, Profit $474.62

QQQ:
  Model:    Invested $22100, Returned $22105.52, Profit $5.52 (221 trades, 282 skipped)
  Buy&Hold: Invested $50300, Returned $50325.43, Profit $25.43
  Perfect:  Invested $26700, Returned $26914.48, Profit $214.48


# Comments

MLP limits losses better than buy & hold across all tickers despite trading less, though CMG and M actually go slightly negative. BB and M avoid the worst of the buy & hold losses. QQQ is the standout, narrowly beating buy & hold with fewer trades. Perfect ceiling remains low at $200-500 on $50k, so the signal just isn't there in the price features alone regardless of mode.